# 🎓 Study Buddy Agent
### A Multi-Agent Adaptive Learning System built with Google ADK

**What it does:** You give it a topic (and optionally a PDF), and it:
1. **Ingests** sources — web snippets + PDF text → chunked knowledge base
2. **Plans** a sequenced study curriculum (Curriculum Agent)
3. **Tutors** — answers questions grounded in the source material (Tutor Agent)
4. **Quizzes** — generates MCQ + open-ended questions, grades with LLM-as-judge (Quiz Agent)
5. **Remembers** — tracks mastery per topic, applies spaced repetition (Memory Store)

---

### Architecture

```
Orchestrator (StudySession)
├── Ingestion tools        ← web search + PDF → SourceChunks
├── Curriculum Agent       ← structured JSON study plan
├── Tutor Agent            ← RAG-lite explanation / Q&A
├── Quiz Generation Agent  ← MCQ + open-ended questions
├── Quiz Grading Agent     ← LLM-as-judge for open-ended scoring
└── Memory Store           ← JSON persistence + spaced repetition
```

All agents are built with **Google ADK** (`google-adk`).
Auth auto-detects between Vertex AI (ADC / `gcloud auth`) and Gemini API key.
In this notebook we use a Gemini API key stored in **Kaggle Secrets**.


In [1]:
# Install dependencies
# google-adk  : agent framework (wraps google-genai)
# pypdf        : PDF text extraction
# googlesearch-python : lightweight web search
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "google-adk", "pypdf", "googlesearch-python"],
    check=True,
)
print("\u2713 Dependencies installed")


✓ Dependencies installed



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 🔑 Auth & Configuration

In [2]:
import os, subprocess

# ── Auth resolution (priority order) ─────────────────────────────────────────
#
#  1. Kaggle Secrets  → API key mode  (for Kaggle notebook submission)
#  2. gcloud ADC      → Vertex AI mode (for local testing with `gcloud auth`)
#  3. GOOGLE_API_KEY env var → API key mode (manual fallback)
#
# You don't need to change anything here — the right path is chosen automatically.

def _adc_available() -> bool:
    """True if Application Default Credentials are configured locally."""
    adc_file = os.environ.get(
        "GOOGLE_APPLICATION_CREDENTIALS",
        os.path.expanduser("~/.config/gcloud/application_default_credentials.json"),
    )
    if os.path.isfile(adc_file):
        return True
    try:
        import google.auth
        creds, _ = google.auth.default()
        return creds is not None
    except Exception:
        pass
    return False


def _gcloud_project() -> str:
    """Read the active gcloud project, or empty string."""
    try:
        r = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=5,
        )
        v = r.stdout.strip()
        return v if v and v != "(unset)" else ""
    except Exception:
        return ""


# ── 1. Kaggle Secrets ─────────────────────────────────────────────────────────
_auth_mode = None
try:
    from kaggle_secrets import UserSecretsClient
    _api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
    os.environ["GOOGLE_API_KEY"] = _api_key
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
    _auth_mode = "api_key (Kaggle Secrets)"
except Exception:
    pass

# ── 2. gcloud Application Default Credentials ────────────────────────────────
if not _auth_mode and _adc_available():
    project = (
        os.environ.get("GOOGLE_CLOUD_PROJECT")
        or _gcloud_project()
    )
    if not project:
        raise EnvironmentError(
            "ADC detected but no GCP project found.\n"
            "Run: gcloud config set project <your-project-id>"
        )
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"
    os.environ.setdefault("GOOGLE_CLOUD_PROJECT", project)
    os.environ.setdefault("GOOGLE_CLOUD_LOCATION", "us-central1")
    _auth_mode = f"vertex_adc (project={os.environ['GOOGLE_CLOUD_PROJECT']})"

# ── 3. GOOGLE_API_KEY env var ─────────────────────────────────────────────────
if not _auth_mode:
    _api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY", "")
    if _api_key:
        os.environ.setdefault("GOOGLE_API_KEY", _api_key)
        os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
        _auth_mode = "api_key (env var)"

if not _auth_mode:
    raise EnvironmentError(
        "No Gemini credentials found. Choose one:\n"
        "  • Locally (gcloud): gcloud auth application-default login\n"
        "  • API key:          export GOOGLE_API_KEY=<key>\n"
        "  • Kaggle:           Add-ons → Secrets → GEMINI_API_KEY"
    )

print(f"\u2713 Auth mode : {_auth_mode}")

# Model selection
MODEL_NAME = "gemini-2.0-flash"
print(f"\u2713 Model     : {MODEL_NAME}")


✓ Auth mode : vertex_adc (project=playground--infosec-play)
✓ Model     : gemini-2.0-flash


## 🤖 Model Selection

In [3]:
# ── Model Discovery ─────────────────────────────────────────────────────────
# Lists Gemini models available to your project. If auto-discovery fails,
# a curated list of known stable Vertex AI models is shown instead.
from google import genai as _genai
import os as _os

_use_vertex = _os.environ.get("GOOGLE_GENAI_USE_VERTEXAI", "False").lower() == "true"
_project    = _os.environ.get("GOOGLE_CLOUD_PROJECT", "")
_location   = _os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")

_discovered = []
try:
    if _use_vertex:
        _c = _genai.Client(vertexai=True, project=_project, location=_location)
    else:
        _c = _genai.Client(api_key=_os.environ.get("GOOGLE_API_KEY", ""))
    for _m in _c.models.list():
        if "gemini" in _m.name.lower() and "gemini-" in _m.name:
            _discovered.append(_m.name)
    print(f"Discovered {len(_discovered)} Gemini model(s) via API")
except Exception as _e:
    print(f"Auto-discovery unavailable ({_e})")

# Curated fallback list of stable Vertex AI Gemini models
_known_vertex = [
    "gemini-1.5-flash",
    "gemini-1.5-flash-002",
    "gemini-1.5-pro",
    "gemini-1.5-pro-002",
    "gemini-2.0-flash-001",
    "gemini-2.0-flash-lite-001",
    "gemini-2.5-flash-preview-05-20",
    "gemini-2.5-pro-preview-06-05",
]

AVAILABLE_MODELS = _discovered if _discovered else _known_vertex
print("\nAvailable Gemini models:")
for _i, _m in enumerate(AVAILABLE_MODELS, 1):
    print(f"  {_i:2}. {_m}")
print("\n→ Set MODEL_NAME in the cell below, then run it.")


Discovered 21 Gemini model(s) via API

Available Gemini models:
   1. publishers/google/models/gemini-1.5-pro-002
   2. publishers/google/models/gemini-2.0-flash-001
   3. publishers/google/models/gemini-2.0-flash-lite-001
   4. publishers/google/models/gemini-2.5-flash-preview-04-17
   5. publishers/google/models/gemini-2.5-pro-exp-03-25
   6. publishers/google/models/gemini-2.5-pro
   7. publishers/google/models/gemini-2.5-flash
   8. publishers/google/models/gemini-2.5-flash-lite
   9. publishers/google/models/gemini-2.5-pro-tts
  10. publishers/google/models/gemini-2.5-flash-tts
  11. publishers/google/models/gemini-live-2.5-flash-native-audio
  12. publishers/google/models/gemini-3-flash-preview
  13. publishers/google/models/gemini-3.1-flash-lite-preview
  14. publishers/google/models/gemini-3.1-flash-image-preview
  15. publishers/google/models/gemini-3.1-pro-preview
  16. publishers/google/models/gemini-3.5-flash
  17. publishers/google/models/gemini-3.1-flash-image
  18. publi

In [4]:
# ── Set your model ───────────────────────────────────────────────────────────
# Pick any model from the list above and paste the name here.
# gemini-1.5-flash is the safest default — available in all regions.
MODEL_NAME = "gemini-2.5-flash"

print(f"✓ Model: {MODEL_NAME}")


✓ Model: gemini-2.5-flash


## 📦 Core Modules
### Memory Store

In [5]:
"""
Persistent memory: tracks per-topic mastery over time so the agent can
apply simple spaced repetition — anything below 0.7 mastery is flagged for review.
"""
import json, os
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Dict, List, Optional

MASTERY_REVIEW_THRESHOLD = 0.7
DECAY_PER_DAY = 0.02
#MEMORY_PATH = "/kaggle/working/study_buddy_memory.json"

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") != "":
    MEMORY_PATH = "/kaggle/working/study_buddy_memory.json"
else:
    MEMORY_PATH = "study_buddy_memory.json"


@dataclass
class TopicRecord:
    subtopics: List[str] = field(default_factory=list)
    scores: List[float] = field(default_factory=list)
    last_reviewed: Optional[str] = None
    mastery: float = 0.0

    def record_score(self, score: float) -> None:
        self.scores.append(score)
        alpha = 0.5
        self.mastery = score if len(self.scores) == 1 else alpha * score + (1 - alpha) * self.mastery
        self.last_reviewed = datetime.now(timezone.utc).date().isoformat()

    def days_since_review(self) -> Optional[int]:
        if not self.last_reviewed:
            return None
        last = datetime.fromisoformat(self.last_reviewed).date()
        return (datetime.now(timezone.utc).date() - last).days

    def effective_mastery(self) -> float:
        days = self.days_since_review()
        decayed = self.mastery - (DECAY_PER_DAY * (days or 0))
        return max(0.0, decayed)


class MemoryStore:
    def __init__(self, path: str = MEMORY_PATH, user: str = "learner"):
        self.path = path
        self.user = user
        self._data: Dict[str, Dict[str, TopicRecord]] = {}
        self._load()

    def _load(self) -> None:
        if not os.path.isfile(self.path):
            self._data = {}
            return
        with open(self.path) as f:
            raw = json.load(f)
        self._data = {
            u: {t: TopicRecord(**r) for t, r in topics.items()}
            for u, topics in raw.items()
        }

    def save(self) -> None:
        os.makedirs(os.path.dirname(self.path) or ".", exist_ok=True)
        with open(self.path, "w") as f:
            json.dump(
                {u: {t: asdict(r) for t, r in topics.items()} for u, topics in self._data.items()},
                f, indent=2,
            )

    def _user_topics(self) -> Dict[str, TopicRecord]:
        return self._data.setdefault(self.user, {})

    def get_topic(self, topic: str) -> TopicRecord:
        topics = self._user_topics()
        if topic not in topics:
            topics[topic] = TopicRecord()
        return topics[topic]

    def upsert_subtopics(self, topic: str, subtopics: List[str]) -> None:
        rec = self.get_topic(topic)
        seen = set(rec.subtopics)
        for s in subtopics:
            if s not in seen:
                rec.subtopics.append(s)
                seen.add(s)

    def record_quiz_score(self, topic: str, score: float) -> TopicRecord:
        rec = self.get_topic(topic)
        rec.record_score(score)
        self.save()
        return rec

    def topics_due_for_review(self) -> List[str]:
        return [t for t, r in self._user_topics().items()
                if r.effective_mastery() < MASTERY_REVIEW_THRESHOLD]

    def summary(self) -> str:
        topics = self._user_topics()
        if not topics:
            return "No study history yet."
        return "\n".join(
            f"  {t}: mastery={r.effective_mastery():.2f} (last reviewed: {r.last_reviewed or 'never'})"
            for t, r in sorted(topics.items(), key=lambda kv: kv[1].effective_mastery())
        )

print("✓ MemoryStore defined")


✓ MemoryStore defined


### Schemas (structured output contracts)

In [6]:
"""
Pydantic schemas used as output_schema= for ADK agents.
Structured output tells the model to return valid JSON matching these shapes.
"""
from pydantic import BaseModel, Field
from typing import List, Optional, Literal


class TopicPlan(BaseModel):
    topic: str
    subtopics: List[str] = Field(description="3-6 specific subtopics to cover")
    rationale: str = Field(description="Why this topic matters for the goal")


class StudyPlan(BaseModel):
    goal_summary: str = Field(description="One-sentence restatement of the learning goal")
    topics: List[TopicPlan] = Field(description="Ordered list of topics (foundational first)")
    estimated_sessions: int = Field(description="Suggested number of study sessions")


class QuizQuestion(BaseModel):
    question: str
    question_type: Literal["multiple_choice", "open_ended"]
    choices: Optional[List[str]] = Field(default=None, description="4 choices for MCQ")
    correct_answer: str
    subtopic: str
    difficulty: Literal["easy", "medium", "hard"]


class QuizSet(BaseModel):
    topic: str
    questions: List[QuizQuestion]


class GradedAnswer(BaseModel):
    is_correct: bool
    score: float = Field(description="0.0-1.0 credit awarded")
    feedback: str = Field(description="Specific, encouraging feedback")

print("✓ Schemas defined")


✓ Schemas defined


### Ingestion Tools

In [7]:
"""
Ingestion: PDF text extraction + web search → SourceChunks for downstream agents.
"""
import logging
from dataclasses import dataclass

logger = logging.getLogger("study_buddy")
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200


@dataclass
class SourceChunk:
    text: str
    source: str


def chunk_text(text: str, source: str) -> List[SourceChunk]:
    text = text.strip()
    if not text:
        return []
    chunks, start = [], 0
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(SourceChunk(text=chunk, source=source))
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks


def ingest_pdf(file_path: str) -> List[SourceChunk]:
    from pypdf import PdfReader
    reader = PdfReader(file_path)
    text = "\n".join(p.extract_text() or "" for p in reader.pages)
    return chunk_text(text, source=file_path)


def web_search_topic(topic: str, num_results: int = 5) -> List[SourceChunk]:
    try:
        from googlesearch import search
    except ImportError:
        logger.warning("googlesearch-python not installed; skipping web search.")
        return []
    chunks = []
    try:
        for r in search(topic, num_results=num_results, advanced=True):
            text = f"{r.title}\n{r.description}".strip()
            if text:
                chunks.append(SourceChunk(text=text, source=r.url))
    except Exception as exc:
        logger.warning("Web search failed: %s", exc)
    return chunks


def build_knowledge_base(
    topic: Optional[str] = None,
    pdf_paths: Optional[List[str]] = None,
    include_web_search: bool = True,
) -> List[SourceChunk]:
    chunks: List[SourceChunk] = []
    for path in (pdf_paths or []):
        try:
            chunks.extend(ingest_pdf(path))
        except Exception as exc:
            logger.error("PDF ingestion failed for %s: %s", path, exc)
    if topic and include_web_search:
        chunks.extend(web_search_topic(topic))
    if not chunks and topic:
        chunks.append(SourceChunk(text=topic, source="user_topic"))
    return chunks


def format_chunks_for_prompt(chunks: List[SourceChunk], max_chars: int = 12000) -> str:
    parts, total = [], 0
    for c in chunks:
        block = f"[Source: {c.source}]\n{c.text}\n"
        if total + len(block) > max_chars:
            break
        parts.append(block)
        total += len(block)
    return "\n---\n".join(parts) if parts else "(no source material available)"

print("✓ Ingestion tools defined")


✓ Ingestion tools defined


### ADK Runner Utilities

In [8]:
"""
Thin synchronous wrapper around ADK InMemoryRunner.
Each call to run_agent_once() creates a fresh session + sends one message.

Fix for newer ADK versions where session_service methods are async:
  - All async work (create_session + run_async) runs inside a single event loop
    created by asyncio.run() in a worker thread, avoiding cross-loop session lookups.
  - runner.run_async() is used instead of runner.run() so session creation and
    agent execution share the same loop.
"""
import asyncio, inspect, json, uuid
from typing import Optional
from google.adk.agents.llm_agent import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types

_APP_NAME = "study_buddy"


async def _run_turn_async(
    agent: Agent,
    message: str,
    user_id: str,
    session_id: str,
) -> str:
    """Session creation + agent execution in one async context = same event loop."""
    runner = InMemoryRunner(agent=agent, app_name=_APP_NAME)
    svc = runner.session_service

    # create_session is async in newer ADK versions
    result = svc.create_session(app_name=_APP_NAME, user_id=user_id, session_id=session_id)
    if inspect.isawaitable(result):
        await result

    content = types.Content(role="user", parts=[types.Part(text=message)])
    parts = []
    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if getattr(p, "text", None):
                    parts.append(p.text)
    return "".join(parts)


def run_agent_turn(
    agent: Agent,
    message: str,
    runner: Optional[InMemoryRunner] = None,  # kept for API compat, no longer used
    user_id: str = "learner",
    session_id: Optional[str] = None,
) -> tuple:
    sid = session_id or str(uuid.uuid4())
    # Run entirely in a fresh thread+loop to avoid Jupyter's running loop conflict.
    import concurrent.futures
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        text = pool.submit(asyncio.run, _run_turn_async(agent, message, user_id, sid)).result()
    return text, sid


def run_agent_once(agent: Agent, message: str, user_id: str = "learner") -> str:
    text, _ = run_agent_turn(agent, message, user_id=user_id)
    return text


def parse_json_output(raw: str):
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:]
    return json.loads(cleaned.strip())

print("✓ Runner utilities defined")

✓ Runner utilities defined


### Agent Definitions

In [9]:
"""
All four agents:
  - Curriculum Agent  : knowledge base → structured StudyPlan JSON
  - Tutor Agent       : answers learner questions with source context
  - Quiz Gen Agent    : generates QuizSet JSON for a topic
  - Quiz Grading Agent: LLM-as-judge for open-ended answers → GradedAnswer JSON
"""
from google.adk.agents.llm_agent import Agent


def build_curriculum_agent(model: str = MODEL_NAME) -> Agent:
    return Agent(
        name="curriculum_agent",
        model=model,
        description="Designs a structured study plan from source material and a learning goal.",
        instruction="""You are a curriculum design specialist. Given source material and a learner's goal,
produce a clear, well-sequenced study plan.

Guidelines:
- Order topics from foundational to advanced.
- Each topic should have 3-6 concrete, specific subtopics.
- Keep the plan realistic — fewer well-chosen topics beats an exhaustive list.
- Base the plan on the source material where possible.
""",
        output_schema=StudyPlan,
        output_key="study_plan",
    )


def build_tutor_agent(model: str = MODEL_NAME) -> Agent:
    return Agent(
        name="tutor_agent",
        model=model,
        description="Explains concepts and answers learner questions, grounded in source material.",
        instruction="""You are a patient, encouraging tutor. Explain concepts clearly using analogies and
examples. Ground explanations in the provided source material when relevant, but
supplement with your own knowledge if needed.

Keep explanations focused — a few well-chosen paragraphs, not an essay.
""",
    )


def build_quiz_generation_agent(model: str = MODEL_NAME) -> Agent:
    return Agent(
        name="quiz_generation_agent",
        model=model,
        description="Generates a mix of MCQ and open-ended quiz questions for a topic.",
        instruction="""You are a quiz designer. Given a topic, its subtopics, and source material,
generate a mix of multiple-choice and open-ended questions that test real
understanding — not just recall of definitions.

Guidelines:
- Vary difficulty: some easier recall questions, some that require applying the concept.
- For MCQ: provide exactly 4 choices with plausible distractors.
- Tag each question with the specific subtopic it tests.
""",
        output_schema=QuizSet,
        output_key="quiz_set",
    )


def build_quiz_grading_agent(model: str = MODEL_NAME) -> Agent:
    return Agent(
        name="quiz_grading_agent",
        model=model,
        description="LLM-as-judge: grades open-ended answers against a reference answer.",
        instruction="""You are grading a learner's answer to an open-ended quiz question. You will be given
the question, the reference/correct answer, and the learner's actual answer.

Grade generously but honestly:
- Full credit (1.0): core concept correctly understood, even if phrasing differs.
- Partial credit (0.3-0.7): some understanding but misses key parts.
- Low credit (0.0-0.2): fundamental misunderstanding or off-topic.

Always give specific, encouraging feedback — note what was right before addressing gaps.
""",
        output_schema=GradedAnswer,
        output_key="graded_answer",
    )


def grade_multiple_choice(selected: str, correct: str) -> GradedAnswer:
    """Deterministic grading for MCQ — no LLM call needed."""
    ok = selected.strip().lower() == correct.strip().lower()
    return GradedAnswer(
        is_correct=ok,
        score=1.0 if ok else 0.0,
        feedback="Correct!" if ok else f"Not quite — the correct answer was: {correct}",
    )

print("✓ All four agents defined")


✓ All four agents defined


### StudySession Orchestrator

In [10]:
"""
StudySession ties together ingestion → curriculum → tutor/quiz → memory.
Agents are built lazily and cached so auth setup only happens once.
"""
from dataclasses import dataclass, field


@dataclass
class StudySession:
    topic: Optional[str] = None
    pdf_paths: List[str] = field(default_factory=list)
    user_id: str = "learner"
    model_name: str = MODEL_NAME

    knowledge_base: List[SourceChunk] = field(default_factory=list, init=False)
    study_plan: Optional[StudyPlan] = field(default=None, init=False)
    memory: MemoryStore = field(init=False)

    def __post_init__(self):
        self.memory = MemoryStore(user=self.user_id)
        self._curriculum = self._tutor = self._quiz_gen = self._quiz_grader = None

    @property
    def curriculum_agent(self):
        if not self._curriculum:
            self._curriculum = build_curriculum_agent(self.model_name)
        return self._curriculum

    @property
    def tutor_agent(self):
        if not self._tutor:
            self._tutor = build_tutor_agent(self.model_name)
        return self._tutor

    @property
    def quiz_gen_agent(self):
        if not self._quiz_gen:
            self._quiz_gen = build_quiz_generation_agent(self.model_name)
        return self._quiz_gen

    @property
    def quiz_grading_agent(self):
        if not self._quiz_grader:
            self._quiz_grader = build_quiz_grading_agent(self.model_name)
        return self._quiz_grader

    # ── Pipeline steps ──────────────────────────────────────────────────────

    def ingest(self) -> List[SourceChunk]:
        self.knowledge_base = build_knowledge_base(
            topic=self.topic, pdf_paths=self.pdf_paths, include_web_search=True
        )
        return self.knowledge_base

    def build_plan(self, learning_goal: Optional[str] = None) -> StudyPlan:
        if not self.knowledge_base:
            self.ingest()
        goal = learning_goal or self.topic or "Understand the material."
        prompt = f"Learning goal: {goal}\n\nSource material:\n{format_chunks_for_prompt(self.knowledge_base)}"
        raw = run_agent_once(self.curriculum_agent, prompt, user_id=self.user_id)
        parsed = parse_json_output(raw)
        self.study_plan = StudyPlan(**parsed)
        for t in self.study_plan.topics:
            self.memory.upsert_subtopics(t.topic, t.subtopics)
        self.memory.save()
        return self.study_plan

    def explain(self, question: str) -> str:
        ctx = format_chunks_for_prompt(self.knowledge_base, max_chars=6000)
        prompt = f"Source material (context):\n{ctx}\n\nLearner's question: {question}"
        return run_agent_once(self.tutor_agent, prompt, user_id=self.user_id)

    def generate_quiz(self, topic: str, num_questions: int = 5) -> QuizSet:
        record = self.memory.get_topic(topic)
        subtopics = ", ".join(record.subtopics) if record.subtopics else "(general coverage)"
        ctx = format_chunks_for_prompt(self.knowledge_base, max_chars=6000)
        prompt = (
            f"Topic: {topic}\nSubtopics to cover: {subtopics}\n"
            f"Number of questions: {num_questions}\n\nSource material:\n{ctx}"
        )
        raw = run_agent_once(self.quiz_gen_agent, prompt, user_id=self.user_id)
        parsed = parse_json_output(raw)
        return QuizSet(**parsed)

    def grade_answer(self, question: QuizQuestion, answer: str) -> GradedAnswer:
        if question.question_type == "multiple_choice":
            return grade_multiple_choice(answer, question.correct_answer)
        prompt = (
            f"Question: {question.question}\n"
            f"Reference answer: {question.correct_answer}\n"
            f"Learner's answer: {answer}"
        )
        raw = run_agent_once(self.quiz_grading_agent, prompt, user_id=self.user_id)
        parsed = parse_json_output(raw)
        return GradedAnswer(**parsed)

    def record_quiz_result(self, topic: str, score: float):
        self.memory.record_quiz_score(topic, score)

    def topics_due_for_review(self) -> List[str]:
        return self.memory.topics_due_for_review()

    def progress_summary(self) -> str:
        return self.memory.summary()

print("✓ StudySession orchestrator defined")
print()
print("Ready to study! All modules loaded.")


✓ StudySession orchestrator defined

Ready to study! All modules loaded.


---
## 🚀 Demo: Studying Transformer Attention Mechanisms

We'll walk through a complete study session on **Transformer architecture and attention mechanisms** — a cornerstone of modern ML. The demo covers all five pipeline stages:

1. **Ingest** web sources about the topic
2. **Generate** a structured study plan
3. **Ask the tutor** a conceptual question
4. **Take a quiz** with MCQ + open-ended questions graded by the LLM judge
5. **Review memory** — mastery tracking and spaced repetition


### Step 1 — Ingest Sources

In [11]:
STUDY_TOPIC = "Transformer architecture and attention mechanisms in deep learning"

session = StudySession(topic=STUDY_TOPIC, user_id="demo_learner")

print(f"🔍 Ingesting sources for: '{STUDY_TOPIC}'")
print("   (searching the web...)")
chunks = session.ingest()

print(f"\n✓ Built knowledge base: {len(chunks)} source chunks")
print("\nSample chunks:")
for c in chunks[:3]:
    print(f"  [{c.source[:60]}...]")
    print(f"  {c.text[:120].strip()}...")
    print()


🔍 Ingesting sources for: 'Transformer architecture and attention mechanisms in deep learning'
   (searching the web...)

✓ Built knowledge base: 1 source chunks

Sample chunks:
  [user_topic...]
  Transformer architecture and attention mechanisms in deep learning...



### Step 2 — Build Study Plan

In [12]:
print("📚 Designing your study plan...")
plan = session.build_plan(learning_goal=STUDY_TOPIC)

print(f"\n🎯 Goal: {plan.goal_summary}")
print(f"   Estimated sessions: {plan.estimated_sessions}")
print()
for i, t in enumerate(plan.topics, 1):
    print(f"Topic {i}: {t.topic}")
    print(f"  Why: {t.rationale}")
    for s in t.subtopics:
        print(f"    • {s}")
    print()


📚 Designing your study plan...

🎯 Goal: The goal is to understand the Transformer architecture and the underlying attention mechanisms in deep learning.
   Estimated sessions: 10

Topic 1: Neural Network Fundamentals & Sequence Modeling Challenges
  Why: Establishes the foundational deep learning context and highlights the problems and limitations that led to the development of Transformers.
    • Brief review of Artificial Neural Networks (ANNs) and Deep Neural Networks (DNNs)
    • Introduction to Recurrent Neural Networks (RNNs) and Long Short-Term Memory (LSTMs)
    • Limitations of RNNs/LSTMs for long sequences (e.g., vanishing gradients, sequential processing bottleneck)
    • Concept of Word Embeddings and their importance for sequence models

Topic 2: Introduction to Attention Mechanisms
  Why: Attention is the core innovation of Transformers; understanding its fundamental concept is essential before diving into its specific implementations.
    • What is Attention? (Intuitive 

### Step 3 — Ask the Tutor

In [13]:
LEARNER_QUESTION = (
    "Can you explain how the scaled dot-product attention works? "
    "I understand there are queries, keys, and values but I'm confused "
    "about why we scale by the square root of the key dimension."
)

print(f"❓ Learner question: {LEARNER_QUESTION}\n")
print("🤔 Tutor is thinking...\n")

explanation = session.explain(LEARNER_QUESTION)

print("📖 Tutor response:")
print("─" * 60)
print(explanation)
print("─" * 60)


❓ Learner question: Can you explain how the scaled dot-product attention works? I understand there are queries, keys, and values but I'm confused about why we scale by the square root of the key dimension.

🤔 Tutor is thinking...

📖 Tutor response:
────────────────────────────────────────────────────────────
That's a great question, and it shows you're really digging into the details of how transformers work! You're absolutely right that queries, keys, and values are fundamental to attention.

Let's break down scaled dot-product attention and then tackle that scaling factor.

Imagine you're at a library trying to find a specific book.
*   Your **query** is what you're looking for (e.g., "a fantasy novel about dragons").
*   The **keys** are the descriptions on each book's spine (e.g., "Fantasy: Dragon Riders," "Sci-Fi: Space Odyssey").
*   The **values** are the actual books themselves, which you'll pick up once you find a good match.

In scaled dot-product attention, we calculate how 

### Step 4 — Quiz Session

This cell shows the LLM-as-judge grading in action:
- **MCQ** is graded deterministically (no extra API call)
- **Open-ended** answers are sent to the Quiz Grading Agent for nuanced scoring


In [14]:
import textwrap

# Pick the first topic from the plan for the quiz
QUIZ_TOPIC = plan.topics[0].topic
NUM_QUESTIONS = 4

print(f"📝 Generating {NUM_QUESTIONS} questions on: '{QUIZ_TOPIC}'")
quiz = session.generate_quiz(QUIZ_TOPIC, num_questions=NUM_QUESTIONS)
print(f"✓ Generated {len(quiz.questions)} questions\n")

# Simulate a learner's answers — mix of correct, partial, and wrong
# (In the real CLI these come from input(); here we hardcode for a clean demo run)
SIMULATED_ANSWERS = {
    "multiple_choice": lambda q: q.choices[0] if q.choices else "A",   # always pick first
    "open_ended": [
        # Answer 1: good answer
        "Attention lets each token look at all other tokens and weight them by relevance, "
        "so the model captures long-range dependencies without recurrence.",
        # Answer 2: partial answer
        "It uses dot products to compare queries and keys, then applies softmax.",
        # Answer 3 (if needed): weak answer
        "It helps the model focus on important words.",
    ],
}

open_ended_answers = iter(SIMULATED_ANSWERS["open_ended"])
total_score = 0.0
results = []

for i, q in enumerate(quiz.questions, 1):
    if q.question_type == "multiple_choice":
        # Simulate picking the correct answer half the time
        answer = q.correct_answer if i % 2 == 0 else (q.choices[0] if q.choices else "A")
    else:
        answer = next(open_ended_answers, "I'm not sure.")

    print(f"Q{i} [{q.difficulty.upper()} | {q.question_type}] {q.question}")
    if q.question_type == "multiple_choice" and q.choices:
        for j, c in enumerate(q.choices, 1):
            marker = "►" if c == answer else " "
            print(f"   {marker} {j}. {c}")
    else:
        print("   Answer given: " + repr(answer))

    graded = session.grade_answer(q, answer)

    emoji = "✅" if graded.is_correct else ("🟡" if graded.score >= 0.5 else "❌")
    print(f"   {emoji} Score: {graded.score:.1f}  |  {graded.feedback}")
    print()
    total_score += graded.score
    results.append((q, answer, graded))

avg_score = total_score / len(quiz.questions)
session.record_quiz_result(QUIZ_TOPIC, avg_score)

print("─" * 60)
print(f"Quiz complete!  Average score: {avg_score:.2f} / 1.00")
print(f"Topic '{QUIZ_TOPIC}' recorded in memory.")


📝 Generating 4 questions on: 'Neural Network Fundamentals & Sequence Modeling Challenges'
✓ Generated 4 questions

Q1 [EASY | multiple_choice] What is the primary characteristic that distinguishes a Deep Neural Network (DNN) from a simple Artificial Neural Network (ANN)?
   ► 1. A) DNNs use only sigmoid activation functions.
     2. B) DNNs have multiple hidden layers.
     3. C) DNNs are exclusively used for image recognition.
     4. D) DNNs do not require training data.
   ❌ Score: 0.0  |  Not quite — the correct answer was: B) DNNs have multiple hidden layers.

Q2 [MEDIUM | open_ended] Explain the fundamental difference in how Recurrent Neural Networks (RNNs) process sequential data compared to a feedforward neural network. Why was this new architecture necessary?
   Answer given: 'Attention lets each token look at all other tokens and weight them by relevance, so the model captures long-range dependencies without recurrence.'
   ❌ Score: 0.0  |  It's great that you're familiar wit

### Step 5 — Memory & Spaced Repetition

In [15]:
# Simulate having studied multiple topics with different scores over several sessions
# (so the mastery/spaced-repetition logic has something interesting to show)
for t in plan.topics:
    topic_name = t.topic
    # Assign synthetic scores: first topic we just quizzed, others get older scores
    if topic_name == QUIZ_TOPIC:
        continue  # already recorded above
    import random; random.seed(42)
    score = random.uniform(0.3, 0.95)
    session.memory.record_quiz_score(topic_name, score)

    # Back-date last_reviewed for some topics to trigger decay
    import json
    from datetime import timedelta
    rec = session.memory.get_topic(topic_name)
    # Simulate the topic was last reviewed 8 days ago
    past_date = (datetime.now(timezone.utc).date() - timedelta(days=8)).isoformat()
    rec.last_reviewed = past_date
session.memory.save()

print("📊 Progress Summary")
print("─" * 60)
print(session.progress_summary())
print()

due = session.topics_due_for_review()
if due:
    print(f"⏰ Topics due for review (mastery < {MASTERY_REVIEW_THRESHOLD}):")
    for t in due:
        rec = session.memory.get_topic(t)
        print(f"   • {t}  (effective mastery: {rec.effective_mastery():.2f})")
else:
    print("✓ All topics above mastery threshold — great work!")

print()
print("💾 Memory persisted to:", session.memory.path)


📊 Progress Summary
────────────────────────────────────────────────────────────
  Neural Network Fundamentals & Sequence Modeling Challenges: mastery=0.00 (last reviewed: 2026-07-07)
  Introduction to Attention Mechanisms: mastery=0.56 (last reviewed: 2026-06-29)
  Scaled Dot-Product and Multi-Head Attention: mastery=0.56 (last reviewed: 2026-06-29)
  The Transformer Encoder Architecture: mastery=0.56 (last reviewed: 2026-06-29)
  The Transformer Decoder Architecture: mastery=0.56 (last reviewed: 2026-06-29)
  Training, Variants, and Applications of Transformers: mastery=0.56 (last reviewed: 2026-06-29)

⏰ Topics due for review (mastery < 0.7):
   • Neural Network Fundamentals & Sequence Modeling Challenges  (effective mastery: 0.00)
   • Introduction to Attention Mechanisms  (effective mastery: 0.56)
   • Scaled Dot-Product and Multi-Head Attention  (effective mastery: 0.56)
   • The Transformer Encoder Architecture  (effective mastery: 0.56)
   • The Transformer Decoder Architecture 

---
## 🏗️ Design Notes & Reflections

### Why multi-agent?

Each agent has a single, testable responsibility. The Curriculum Agent only designs plans; the Quiz Grading Agent only scores answers. This separation makes it easy to:
- Swap prompts or models per agent without breaking others
- Unit-test each agent independently (mock the LLM, assert on schema)
- Explain the system to non-technical stakeholders using the agent names

The Orchestrator (`StudySession`) is plain Python — not an ADK Workflow — because the top-level session loop depends on real user input, which maps more naturally to an explicit `while True` than a static graph.

### LLM-as-judge (Day 4 eval concept)

Multiple-choice questions are graded deterministically (zero cost, zero latency, zero ambiguity). Open-ended questions are sent to a separate `quiz_grading_agent` with a rubric-style instruction. This is the LLM-as-judge eval pattern: using a second model call to score a first model's (or a human's) output against a reference.

### Memory & spaced repetition

`MemoryStore` persists a JSON file of `topic → TopicRecord`, where each record tracks:
- quiz score history (exponential moving average → `mastery`)
- `last_reviewed` date
- a per-day decay (`mastery -= 0.02/day`)

Any topic whose `effective_mastery` falls below 0.7 is flagged for review at session start. This is a deliberately simple implementation of the [Leitner / spaced repetition](https://en.wikipedia.org/wiki/Spaced_repetition) principle — good enough to demonstrate adaptive behavior without pulling in a heavy SRS library.

### Auth design

The `GOOGLE_GENAI_USE_VERTEXAI` environment variable controls which auth mode `google-genai` (ADK's underlying library) uses. We set it to `"False"` here (API key path). The standalone application version (`study_buddy/config/models.py`) auto-detects between Vertex AI ADC and API key, making `gcloud auth application-default login` the zero-config local experience.

### What I'd improve next

- **Vector search** — swap the flat chunk list for a proper embedding index (e.g. FAISS or ChromaDB) to do semantic retrieval instead of just passing all chunks to every prompt.
- **PDF upload via Gemini Files API** — for large documents, use `google.generativeai.upload_file()` so the full document lives server-side and doesn't eat context window.
- **Persistent ADK sessions** — replace `InMemoryRunner` with a `DatabaseSessionService` so tutor conversations survive across notebook runs.
- **Confidence calibration** — use the grading agent's score history to detect when the model is systematically over- or under-grading, and adjust the rubric accordingly.
